
# AI IN HEALTHCARE: High Risk Project COLAB NOTEBOOK
# Mohsin Imam
# Topic: Translator


In [2]:
!pip install -q -U google-generativeai pandas

In [17]:
import pandas as pd
import google.generativeai as genai
import json
import time
from google.colab import userdata

# --- 1. SETUP GEMINI API ---
genai.configure(
    api_key=userdata.get('GEMINI_API_KEY'),
    client_options={"api_endpoint": "generativelanguage.googleapis.com"} # Explicitly set API endpoint
)
# We use JSON mode to ensure the output is perfectly structured for our training dataset
model = genai.GenerativeModel(
    'gemini-2.5-flash',
    generation_config={"response_mime_type": "application/json"}
)

In [18]:
# --- 2. LOAD SYNTHEA DATA ---
print("Loading Synthea CSVs...")
try:
    patients_df = pd.read_csv('patients.csv')
    conditions_df = pd.read_csv('conditions.csv')
    medications_df = pd.read_csv('medications.csv')
except FileNotFoundError:
    print("Error: Please make sure patients.csv, conditions.csv, and medications.csv are uploaded to Colab.")

# --- 3. STITCH TABULAR DATA INTO CLINICAL NOTES ---
print("Generating synthetic clinical notes...")
patient_ids = patients_df['Id'].head(5).tolist()
synthetic_notes = []

for pid in patient_ids:
    # Extract unique conditions and medications for the specific patient
    pt_conditions = conditions_df[conditions_df['PATIENT'] == pid]['DESCRIPTION'].dropna().unique().tolist()
    pt_meds = medications_df[medications_df['PATIENT'] == pid]['DESCRIPTION'].dropna().unique().tolist()

    # Skip patients with no medical history to ensure our training data is robust
    if not pt_conditions:
        continue

    # Construct the jargon-heavy note
    note = f"Patient ID {pid[:8]} presents with a documented medical history significant for {', '.join(pt_conditions)}. "
    if pt_meds:
        note += f"Current pharmacological interventions include: {', '.join(pt_meds)}. "
    note += "Plan: Continue current care regimen, monitor for therapeutic efficacy and adverse interactions. Follow up in outpatient clinic."

    print(note)
    synthetic_notes.append(note)

notes_df = pd.DataFrame({'clinical_note': synthetic_notes})
print(f"Successfully generated {len(notes_df)} clinical notes.\n")

# --- 4. GENERATE GROUND TRUTH (TEACHER MODEL) ---
print("Calling Gemini API to generate English & Urdu ground truth...")
training_data = []

# We'll process a small batch first to ensure everything works perfectly
for index, row in notes_df.iterrows():
    print(f"Translating note {index + 1}/{len(notes_df)}...")

    prompt = f"""
    You are an expert medical translator. Read the following clinical note and output exactly two things:
    1. "simple_english": A simplified summary written at an 8th-grade reading level, removing complex jargon but keeping clinical accuracy.
    2. "urdu_translation": A highly accurate, culturally appropriate Urdu translation of the simplified English summary.

    Clinical Note: {row['clinical_note']}
    """

    try:
        response = model.generate_content(prompt, request_options={"timeout": 60}) # Added timeout
        print(f"API Response for note {index + 1}: {response.text[:100]}...") # Log partial response
        result = json.loads(response.text)

        # Structure it exactly how the Gemini Fine-Tuning API expects it
        training_data.append({
            "text_input": row['clinical_note'],
            "output": f"Simple English: {result['simple_english']}\n\nUrdu: {result['urdu_translation']}"
        })

        # Sleep to respect API rate limits
        time.sleep(3)

    except Exception as e:
        print(f"Error processing note {index + 1}: {e}")

# --- 5. SAVE DATASET ---
# Convert the training data dictionary into a DataFrame to inspect and save
training_df = pd.DataFrame(training_data)
training_df.to_csv('fine_tuning_dataset.csv', index=False)

print("\nPipeline Complete! Here is a preview of your training data:")
print(training_df.head())

Loading Synthea CSVs...
Generating synthetic clinical notes...
Patient ID f1aa52b9 presents with a documented medical history significant for Housing unsatisfactory (finding), Received higher education (finding), Loss of teeth (disorder), Full-time employment (finding), Acute bronchitis (disorder), Medication review due (situation), Unemployed (finding), Viral sinusitis (disorder), Reports of violence in the environment (finding), Stress (finding), Acute viral pharyngitis (disorder), Gingivitis (disorder), Gingival disease (disorder). Current pharmacological interventions include: Acetaminophen 325 MG Oral Tablet, Amoxicillin 250 MG / Clavulanate 125 MG Oral Tablet, sodium fluoride 0.0272 MG/MG Oral Gel. Plan: Continue current care regimen, monitor for therapeutic efficacy and adverse interactions. Follow up in outpatient clinic.
Patient ID d30ace70 presents with a documented medical history significant for Received higher education (finding), Prediabetes (finding), Stress (finding), B

In [23]:
import google.generativeai as genai
import random
import time
from google.colab import userdata

# Ensure API is configured with the correct endpoint for fine-tuning
genai.configure(
    api_key=userdata.get('GEMINI_API_KEY'),
    client_options={"api_endpoint": "generativelanguage.googleapis.com"}
)

# 1. Name your custom model (needs to be unique)
# We use a random integer to avoid naming collisions if you run this multiple times
tuned_model_name = f"health-translator-flash-{random.randint(1000,9999)}"

print(f"Starting fine-tuning job for: {tuned_model_name}")
print("This will take a little while. Google's servers are doing the heavy lifting. ☕\n")

# 2. Kick off the fine-tuning job
# Note: We use 'gemini-1.5-flash-001-tuning' as the base model specifically meant for tuning
try:
    operation = genai.create_tuned_model(
        display_name=tuned_model_name,
        source_model="models/gemini-2.5-flash-001-tuning",
        epoch_count=5,
        batch_size=4,
        learning_rate=0.001,
        training_data=training_data # Passing the list of dicts from our previous script
    )

    # 3. Monitor the progress
    for status in operation.wait_bar():
        time.sleep(10)

    print("\nFine-tuning complete! 🎉")

    # 4. Save your new model's permanent ID
    my_custom_model_id = operation.result().name
    print(f"Your model's permanent ID is: {my_custom_model_id}")

except Exception as e:
    print(f"An error occurred during fine-tuning: {e}")

Starting fine-tuning job for: health-translator-flash-4288
This will take a little while. Google's servers are doing the heavy lifting. ☕

An error occurred during fine-tuning: 501 POST https://generativelanguage.googleapis.com/v1beta/tunedModels?%24alt=json%3Benum-encoding%3Dint: Operation is not implemented, or supported, or enabled.
